# Gradient norm tracking in PyTorch

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/klea-ziu/klea-ziu.github.io/blob/master/notebooks/gradient_issues_foundation_models_demo.ipynb)

This notebook is a cleaned Colab version of the original gradient-tracking demo. It uses the GLUE MRPC dataset and a small BERT-style sequence-classification model, then tracks gradient norms during fine-tuning.

There is no API token or external dashboard required. Metrics are stored locally in Python and plotted with matplotlib. If you later create a real experiment-tracking project, the same metric dictionary can be sent to a logger of your choice.

The Hugging Face dependencies and public model, tokenizer, and dataset revisions are pinned so the notebook remains reproducible. A free Colab CPU runtime is sufficient; a GPU only makes the short training runs faster.


## 1. Install dependencies

Colab includes PyTorch, NumPy, and matplotlib. The Hugging Face libraries are pinned to versions tested end to end with this notebook.


In [ ]:
%pip -q install "transformers==5.13.1" "datasets==5.0.0"


## 2. Imports and configuration

In [ ]:
import math
import random
from collections import defaultdict

import datasets
import matplotlib.pyplot as plt
import numpy as np
import torch
import transformers
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch {torch.__version__}, Transformers {transformers.__version__}, Datasets {datasets.__version__}")

# Public resources are pinned to immutable revisions for reproducibility.
MODEL_NAME = "google/bert_uncased_L-2_H-128_A-2"
MODEL_REVISION = "30b0a37ccaaa32f332884b96992754e246e48c5f"
TOKENIZER_NAME = "google-bert/bert-base-uncased"
TOKENIZER_REVISION = "86b5e0934494bd15c9632b12f734a8a67f723594"
DATASET_NAME = "nyu-mll/glue"
DATASET_CONFIG = "mrpc"
DATASET_REVISION = "bcdcba79d07bc864c1c254ccfcedcce55bcc9a8c"

TRAIN_SAMPLES = 512
EVAL_SAMPLES = 256
BATCH_SIZE = 8
MAX_LENGTH = 128
MAX_STEPS = 40


## 3. Load and tokenize MRPC

MRPC is a sentence-pair classification task from the namespaced `nyu-mll/glue` dataset repository. The goal is to predict whether two sentences are paraphrases.


In [ ]:
dataset = load_dataset(DATASET_NAME, DATASET_CONFIG, revision=DATASET_REVISION)
tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_NAME, revision=TOKENIZER_REVISION
)


def tokenize_function(examples):
    return tokenizer(
        examples["sentence1"],
        examples["sentence2"],
        truncation=True,
        max_length=MAX_LENGTH,
    )


tokenized = dataset.map(tokenize_function, batched=True)
tokenized = tokenized.rename_column("label", "labels")

columns = ["input_ids", "attention_mask", "labels"]
if "token_type_ids" in tokenized["train"].features:
    columns.append("token_type_ids")
tokenized.set_format(type="torch", columns=columns)

train_count = min(TRAIN_SAMPLES, len(tokenized["train"]))
eval_count = min(EVAL_SAMPLES, len(tokenized["validation"]))
train_dataset = tokenized["train"].shuffle(seed=SEED).select(range(train_count))
eval_dataset = tokenized["validation"].shuffle(seed=SEED).select(range(eval_count))

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
train_dataloader = DataLoader(
    train_dataset,
    shuffle=True,
    batch_size=BATCH_SIZE,
    collate_fn=data_collator,
)
eval_dataloader = DataLoader(
    eval_dataset,
    batch_size=BATCH_SIZE,
    collate_fn=data_collator,
)

print(f"Training examples: {len(train_dataset)}; validation examples: {len(eval_dataset)}")
print(train_dataset[0].keys())


## 4. Gradient metric helpers

The original notebook logged every parameter gradient to an experiment tracker. Here we keep the same idea but store metrics locally.

For readability, gradients are grouped into embeddings, encoder layers, classifier, and other parameters. This is similar to how you would monitor a larger transformer by block.

In [ ]:
def total_grad_norm(model):
    total = 0.0
    for param in model.parameters():
        if param.grad is not None:
            value = param.grad.detach().float().norm(2).item()
            total += value * value
    return math.sqrt(total)


def parameter_group(name):
    if "embeddings" in name:
        return "embeddings"
    if "encoder.layer" in name:
        parts = name.split("encoder.layer.", 1)[1].split(".")
        layer_id = int(parts[0])
        return f"encoder.layer.{layer_id:02d}"
    if "classifier" in name:
        return "classifier"
    return "other"


def grouped_gradient_norms(model):
    grouped = defaultdict(float)
    for name, param in model.named_parameters():
        if param.grad is None:
            continue
        value = param.grad.detach().float().norm(2).item()
        grouped[parameter_group(name)] += value * value
    return {key: math.sqrt(value) for key, value in grouped.items()}


def move_batch_to_device(batch):
    return {key: value.to(device) for key, value in batch.items()}

## 5. Training loop with local logging

Each step logs:

- training loss,
- total gradient norm,
- grouped gradient norms by model component.

The `clip_norm` argument is optional. Use it to compare unclipped and clipped training.

In [ ]:
def train_and_track(run_name, learning_rate=5e-5, clip_norm=None, max_steps=MAX_STEPS):
    torch.manual_seed(SEED)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2, revision=MODEL_REVISION
    ).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    history = []

    model.train()
    global_step = 0
    while global_step < max_steps:
        for batch in train_dataloader:
            batch = move_batch_to_device(batch)
            optimizer.zero_grad(set_to_none=True)

            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()

            group_norms = grouped_gradient_norms(model)
            total_norm = total_grad_norm(model)

            if clip_norm is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_norm)

            finite = math.isfinite(loss.item()) and math.isfinite(total_norm)
            history.append(
                {
                    "run": run_name,
                    "step": global_step,
                    "loss": loss.item(),
                    "total_grad_norm": total_norm,
                    **{f"gradients/{key}": value for key, value in group_norms.items()},
                }
            )

            if not finite:
                print(f"{run_name}: stopping at step {global_step}; loss or gradients became non-finite")
                return history

            optimizer.step()
            global_step += 1
            if global_step >= max_steps:
                break

    return history

## 6. Run stable and intentionally aggressive settings

The stable run mirrors a normal fine-tuning learning rate. The aggressive run uses a deliberately high learning rate to make instability easier to see. The clipped run shows how gradient clipping can limit the update scale.

Transformers reports that the task-specific classifier weights are newly initialized. This is expected: the checkpoint contains pretrained BERT weights, while the two-class MRPC head is learned during this demo.


In [ ]:
stable_history = train_and_track("stable_lr", learning_rate=5e-5, clip_norm=None)
aggressive_history = train_and_track("aggressive_lr", learning_rate=5e-3, clip_norm=None)
clipped_history = train_and_track("aggressive_lr_clipped", learning_rate=5e-3, clip_norm=1.0)

all_runs = {
    "stable_lr": stable_history,
    "aggressive_lr": aggressive_history,
    "aggressive_lr_clipped": clipped_history,
}

for name, records in all_runs.items():
    assert len(records) == MAX_STEPS, f"{name} stopped after {len(records)} steps"
    assert all(
        math.isfinite(row["loss"]) and math.isfinite(row["total_grad_norm"])
        for row in records
    ), f"{name} produced a non-finite metric"
    last = records[-1]
    print(
        f"{name}: steps={len(records)}, final loss={last['loss']:.4f}, "
        f"final total_grad_norm={last['total_grad_norm']:.4f}"
    )


## 7. Plot loss and total gradient norm

In [ ]:
def series(records, key):
    return [row[key] for row in records]


fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for name, records in all_runs.items():
    axes[0].plot(series(records, "step"), series(records, "loss"), label=name)
    axes[1].plot(series(records, "step"), series(records, "total_grad_norm"), label=name)

axes[0].set_title("Training loss")
axes[0].set_xlabel("step")
axes[0].set_ylabel("loss")
axes[0].grid(True, alpha=0.3)

axes[1].set_title("Total gradient norm")
axes[1].set_xlabel("step")
axes[1].set_ylabel("L2 norm")
axes[1].set_yscale("log")
axes[1].grid(True, alpha=0.3)
axes[1].legend(fontsize=8)
plt.tight_layout()

## 8. Plot gradient norms by transformer block

This plot helps identify where gradient instability appears. In larger models, this is often more useful than a single global norm.

In [ ]:
def plot_grouped_gradients(records, run_name):
    gradient_keys = sorted({key for row in records for key in row if key.startswith("gradients/")})
    data = np.array([[row.get(key, np.nan) for key in gradient_keys] for row in records], dtype=float)
    data = np.log10(np.nan_to_num(data, nan=1e-12) + 1e-12)

    plt.figure(figsize=(12, 4.5))
    plt.imshow(data.T, aspect="auto", interpolation="nearest", cmap="viridis")
    plt.colorbar(label="log10 gradient norm")
    plt.yticks(range(len(gradient_keys)), [key.replace("gradients/", "") for key in gradient_keys])
    plt.xlabel("training step")
    plt.title(f"Grouped gradient norms: {run_name}")
    plt.tight_layout()


plot_grouped_gradients(stable_history, "stable_lr")
plot_grouped_gradients(aggressive_history, "aggressive_lr")
plot_grouped_gradients(clipped_history, "aggressive_lr_clipped")

## 9. Optional: send these metrics to a real tracker

This notebook intentionally avoids external logging because there is no public tracking project attached to the blog post. If you later create a real project, log each dictionary in `history` to your tracker of choice.

Pseudo-code:

```python
for row in stable_history:
    step = row["step"]
    metrics = {key: value for key, value in row.items() if key not in {"run", "step"}}
    tracker.log(metrics, step=step)
```

The important part is the metric naming convention: `loss`, `total_grad_norm`, and `gradients/<component>`.